# TI Support RAG — integración HTTP con Groq

Cambio principal: la integración con Groq se realiza mediante **HTTP directo con `requests.Session`**, en lugar del cliente `Groq` del SDK. La finalidad es aislar y controlar la capa HTTP y medir por separado la latencia del cliente y la latencia reportada por Groq.

In [2]:
from pathlib import Path
import json
import os
import time
import requests
from dotenv import load_dotenv
from datetime import datetime, timezone

file = "system_v2.md"

def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "prompts" / file).exists():
            return path
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto TI-support-RAG."
    )

PROJECT_ROOT = find_project_root()
PROMPT_PATH = PROJECT_ROOT / "prompts" / file
TEST_PATH = PROJECT_ROOT / "test" / "test_cases.json"
RESULTS_PATH = PROJECT_ROOT / "docs" / "results.json"

MODEL = "openai/gpt-oss-20b"
PROMPT_VERSION = "system_v2"
API_URL = "https://api.groq.com/openai/v1/chat/completions"

print(f"Proyecto:   {PROJECT_ROOT}")
print(f"Prompt:     {PROMPT_PATH}")
print(f"Casos:      {TEST_PATH}")
print(f"Resultados: {RESULTS_PATH}")
print(f"Modelo:     {MODEL}")
print(f"Versión:    {PROMPT_VERSION}")
print(f"Endpoint:   {API_URL}")

Proyecto:   /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG
Prompt:     /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/prompts/system_v2.md
Casos:      /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/test/test_cases.json
Resultados: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/docs/results.json
Modelo:     openai/gpt-oss-20b
Versión:    system_v2
Endpoint:   https://api.groq.com/openai/v1/chat/completions


In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError("No se encontró GROQ_API_KEY en el archivo .env.")

print("GROQ_API_KEY configurada: True")

GROQ_API_KEY configurada: True


In [4]:
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

with open(TEST_PATH, "r", encoding="utf-8") as f:
    TEST_CASES = json.load(f)

print(f"Prompt cargado: {len(SYSTEM_PROMPT)} caracteres")
print(f"Casos de prueba cargados: {len(TEST_CASES)}")

for case in TEST_CASES:
    print(f"- {case['id']}: {case['tipo']}")

Prompt cargado: 25333 caracteres
Casos de prueba cargados: 5
- case_01: normal
- case_02: ambiguo
- case_03: incompleto
- case_04: malicioso
- case_05: fuera_de_alcance


In [5]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from schemas.request_v1 import SolicitudTI
from pydantic import ValidationError

def validate_output(data):
    try:
        validated = SolicitudTI.model_validate(data)
        return True, [], validated
    except ValidationError as error:
        errors = []
        for item in error.errors():
            location = ".".join(str(part) for part in item["loc"])
            errors.append(f"{location}: {item['msg']}")
        return False, errors, None

print("Schema SolicitudTI y validación Pydantic cargados correctamente.")

Schema SolicitudTI y validación Pydantic cargados correctamente.


In [6]:
# Integración HTTP directa.
# Session permite reutilizar conexiones TCP/TLS mediante keep-alive.
# El timeout queda explícito para evitar esperas indefinidas.

HTTP_TIMEOUT = (10, 60)  # connect, read (segundos)

http = requests.Session()
http.headers.update({
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Groq-Beta": "inference-metrics",
})

print("Cliente HTTP inicializado.")
print(f"Timeout connect/read: {HTTP_TIMEOUT}")

Cliente HTTP inicializado.
Timeout connect/read: (10, 60)


In [7]:
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "solicitud_ti",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "categoria": {
                    "type": "string",
                    "enum": [
                        "hardware", "software", "redes", "cuentas",
                        "seguridad", "acceso", "otros",
                    ],
                },
                "prioridad": {
                    "type": "string",
                    "enum": ["baja", "media", "alta"],
                },
                "resumen": {"type": "string"},
                "datos_faltantes": {
                    "type": "array",
                    "items": {"type": "string"},
                },
                "requiere_humano": {"type": "boolean"},
                "confianza": {
                    "type": "number",
                    "minimum": 0.0,
                    "maximum": 1.0,
                },
            },
            "required": [
                "categoria", "prioridad", "resumen",
                "datos_faltantes", "requiere_humano", "confianza",
            ],
            "additionalProperties": False,
        },
    },
}

SYSTEM_PROMPT_HTTP = (
    SYSTEM_PROMPT
    + "\n\n"
    + "IMPORTANTE SOBRE EL CAMPO 'confianza':\n"
    + "- Debe ser un número decimal JSON entre 0.0 y 1.0.\n"
    + "- Debe enviarse como número, nunca como cadena de texto.\n"
    + "- Correcto: 0.95\n"
    + "- Incorrecto: \"0.95\""
)

print("Esquema JSON estructurado preparado.")

Esquema JSON estructurado preparado.


In [15]:
import re


def _safe_headers(response):
    interesting = [
        "x-groq-region",
        "cf-ray",
        "retry-after",
        "x-ratelimit-limit-tokens",
        "x-ratelimit-remaining-tokens",
        "x-ratelimit-reset-tokens",
        "x-ratelimit-limit-requests",
        "x-ratelimit-remaining-requests",
        "x-ratelimit-reset-requests",
    ]

    return {
        key: response.headers.get(key)
        for key in interesting
        if response.headers.get(key) is not None
    }


def _get_retry_delay(response, error_body, default_delay=15):
    """
    Determina cuánto tiempo esperar antes de reintentar una solicitud 429.

    Prioridad:
    1. Header Retry-After
    2. Tiempo indicado en el mensaje de Groq
    3. default_delay
    """

    # 1. Header Retry-After
    retry_after = response.headers.get("retry-after")

    if retry_after:
        try:
            return max(float(retry_after), 0)
        except ValueError:
            pass

    # 2. Buscar "Please try again in 15.614999999s"
    error_text = str(error_body)

    match = re.search(
        r"try again in\s+([0-9]+(?:\.[0-9]+)?)s",
        error_text,
        re.IGNORECASE,
    )

    if match:
        return max(float(match.group(1)), 0)

    # 3. Valor por defecto
    return default_delay


def call_groq(user_text, max_retries=3):
    payload = {
        "model": MODEL,
        "temperature": 0,
        "max_completion_tokens": 1500,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT_HTTP,
            },
            {
                "role": "user",
                "content": (
                    "<texto_usuario>\n"
                    f"{user_text}\n"
                    "</texto_usuario>"
                ),
            },
        ],
        "response_format": RESPONSE_SCHEMA,
    }

    start_time = time.perf_counter()
    attempts = 0
    retry_delays = []

    while attempts <= max_retries:

        attempts += 1

        try:
            response = http.post(
                API_URL,
                json=payload,
                timeout=HTTP_TIMEOUT,
            )

            response_headers = _safe_headers(response)

            # ---------------------------------------------------------
            # Manejo específico de HTTP 429
            # ---------------------------------------------------------
            if response.status_code == 429:

                try:
                    error_body = response.json()
                except ValueError:
                    error_body = response.text

                if attempts > max_retries:
                    elapsed = time.perf_counter() - start_time

                    raise RuntimeError(
                        f"HTTP 429 después de {attempts} intentos: "
                        f"{error_body}"
                    )

                retry_delay = _get_retry_delay(
                    response=response,
                    error_body=error_body,
                )

                retry_delays.append(retry_delay)

                print(
                    f"HTTP 429 - límite de tasa alcanzado. "
                    f"Reintentando en {retry_delay:.2f} segundos "
                    f"(intento {attempts}/{max_retries + 1})..."
                )

                time.sleep(retry_delay)

                continue

            # ---------------------------------------------------------
            # Otros errores HTTP
            # ---------------------------------------------------------
            if not response.ok:

                try:
                    error_body = response.json()
                except ValueError:
                    error_body = response.text

                elapsed = time.perf_counter() - start_time

                raise RuntimeError(
                    f"HTTP {response.status_code}: {error_body}"
                )

            # ---------------------------------------------------------
            # Respuesta exitosa
            # ---------------------------------------------------------
            elapsed = time.perf_counter() - start_time

            data = response.json()

            choice = data["choices"][0]
            usage_data = data.get("usage") or {}

            usage = {
                "prompt_tokens": usage_data.get("prompt_tokens"),
                "completion_tokens": usage_data.get("completion_tokens"),
                "total_tokens": usage_data.get("total_tokens"),
            }

            metrics = {
                "queue_time": usage_data.get("queue_time"),
                "prompt_time": usage_data.get("prompt_time"),
                "completion_time": usage_data.get("completion_time"),
                "total_time": usage_data.get("total_time"),
                "cached_tokens": None,
            }

            prompt_tokens_details = usage_data.get(
                "prompt_tokens_details"
            )

            if isinstance(prompt_tokens_details, dict):
                metrics["cached_tokens"] = (
                    prompt_tokens_details.get("cached_tokens")
                )

            server_total = metrics.get("total_time")

            network_overhead = None

            if server_total is not None:
                network_overhead = round(
                    elapsed - float(server_total),
                    4,
                )

            return {
                "raw_output": choice["message"].get("content"),
                "finish_reason": choice.get("finish_reason"),
                "latency_seconds": round(elapsed, 4),
                "usage": usage,
                "metrics": metrics,
                "http": {
                    "status_code": response.status_code,
                    "url": API_URL,
                    "network_overhead_seconds": network_overhead,
                    "headers": response_headers,
                    "attempts": attempts,
                    "retry_delays": retry_delays,
                },
            }

        except requests.RequestException as error:

            elapsed = time.perf_counter() - start_time

            raise RuntimeError(
                f"Error HTTP después de {elapsed:.4f} s: {error}"
            ) from error


print("Función call_groq() HTTP definida correctamente.")

Función call_groq() HTTP definida correctamente.


In [16]:
# Prueba de integración HTTP mínima.
# Esta prueba no utiliza el prompt largo del sistema.

MINIMAL_PAYLOAD = {
    "model": MODEL,
    "temperature": 0,
    "max_completion_tokens": 20,
    "messages": [
        {"role": "user", "content": "Responde únicamente: OK"}
    ],
}

start = time.perf_counter()

response = http.post(
    API_URL,
    json=MINIMAL_PAYLOAD,
    timeout=HTTP_TIMEOUT,
)

elapsed = time.perf_counter() - start

print("HTTP status:", response.status_code)
print("Latencia cliente:", round(elapsed, 4), "s")

try:
    minimal_data = response.json()
    print("Respuesta:", minimal_data["choices"][0]["message"]["content"])
    print("Groq total_time:", minimal_data.get("usage", {}).get("total_time"), "s")
except ValueError:
    print("Respuesta no JSON:")
    print(response.text[:1000])

print("\nHeaders de diagnóstico:")
for key, value in _safe_headers(response).items():
    print(f"{key}: {value}")

HTTP status: 200
Latencia cliente: 0.3889 s
Respuesta: 
Groq total_time: 0.033045688 s

Headers de diagnóstico:
x-groq-region: msp
cf-ray: a397b4368b4b3e9c-BOG
x-ratelimit-limit-tokens: 8000
x-ratelimit-remaining-tokens: 7904
x-ratelimit-reset-tokens: 720ms
x-ratelimit-limit-requests: 1000
x-ratelimit-remaining-requests: 999
x-ratelimit-reset-requests: 1m26.4s


In [17]:
test_response = call_groq(
    "Mi computador no enciende desde esta mañana."
)

print("SALIDA RAW:")
print(test_response["raw_output"])

print("\nRAZÓN DE FINALIZACIÓN:")
print(test_response["finish_reason"])

print("\nLATENCIA CLIENTE:")
print(test_response["latency_seconds"], "segundos")

print("\nMÉTRICAS GROQ:")
print(test_response["metrics"])

print("\nOVERHEAD CLIENTE - SERVIDOR:")
print(test_response["http"]["network_overhead_seconds"], "segundos")

print("\nUSO DE TOKENS:")
print(test_response["usage"])

print("\nHTTP:")
print("Status:", test_response["http"]["status_code"])
print("Headers:", test_response["http"]["headers"])

SALIDA RAW:
{"categoria":"hardware","prioridad":"media","resumen":"El computador del usuario no enciende desde la mañana.","datos_faltantes":["Modelo del equipo","Estado de la batería"],"requiere_humano":true,"confianza":0.9}

RAZÓN DE FINALIZACIÓN:
stop

LATENCIA CLIENTE:
4.7996 segundos

MÉTRICAS GROQ:
{'queue_time': 0.180772079, 'prompt_time': 0.372111719, 'completion_time': 0.967371488, 'total_time': 1.339483207, 'cached_tokens': None}

OVERHEAD CLIENTE - SERVIDOR:
3.4601 segundos

USO DE TOKENS:
{'prompt_tokens': 6268, 'completion_tokens': 898, 'total_tokens': 7166}

HTTP:
Status: 200
Headers: {'x-groq-region': 'msp', 'cf-ray': 'a397b4644d213e9c-BOG', 'x-ratelimit-limit-tokens': '8000', 'x-ratelimit-remaining-tokens': '1712', 'x-ratelimit-reset-tokens': '47.16s', 'x-ratelimit-limit-requests': '1000', 'x-ratelimit-remaining-requests': '998', 'x-ratelimit-reset-requests': '2m52.8s'}


In [18]:
parsed_response = json.loads(test_response["raw_output"])

validation_ok, validation_errors, validated_output = validate_output(
    parsed_response
)

print("VALIDACIÓN:", validation_ok)

if validation_errors:
    print("\nERRORES ENCONTRADOS:")
    for error in validation_errors:
        print(f"- {error}")
else:
    print("\nLa salida cumple el contrato.")
    print("\nSALIDA VALIDADA:")
    print(validated_output.model_dump())

VALIDACIÓN: True

La salida cumple el contrato.

SALIDA VALIDADA:
{'categoria': 'hardware', 'prioridad': 'media', 'resumen': 'El computador del usuario no enciende desde la mañana.', 'datos_faltantes': ['Modelo del equipo', 'Estado de la batería'], 'requiere_humano': True, 'confianza': 0.9}


In [21]:
def run_case(case):
    result = {
        "id": case["id"],
        "tipo": case["tipo"],
        "input": case["input"],
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "raw_output": None,
        "parsed_output": None,
        "validated_output": None,
        "validation_ok": False,
        "validation_errors": [],
        "technical_error": None,
        "decision": None,
        "latency_seconds": None,
        "usage": None,
        "metrics": None,
        "http": None,
        "finish_reason": None,
    }

    try:
        response = call_groq(case["input"])
        result["raw_output"] = response["raw_output"]
        result["latency_seconds"] = response["latency_seconds"]
        result["usage"] = response["usage"]
        result["metrics"] = response["metrics"]
        result["http"] = response["http"]
        result["finish_reason"] = response["finish_reason"]

        try:
            parsed_output = json.loads(response["raw_output"])
        except json.JSONDecodeError as error:
            result["technical_error"] = f"JSON inválido: {error}"
            result["decision"] = "ERROR_TECNICO"
            return result

        result["parsed_output"] = parsed_output

        validation_ok, validation_errors, validated_output = validate_output(
            parsed_output
        )

        result["validation_ok"] = validation_ok
        result["validation_errors"] = validation_errors

        if validated_output is not None:
            result["validated_output"] = validated_output.model_dump()

        result["decision"] = (
            "OK_VALIDADO" if validation_ok else "ERROR_FORMATO"
        )

    except Exception as error:
        result["technical_error"] = str(error)
        result["decision"] = "ERROR_TECNICO"

    return result

print("Funcion de prueba de casos completada")

Funcion de prueba de casos completada


In [25]:
import random

case = random.choice(TEST_CASES)

print("=" * 60)
print("CASO SELECCIONADO ALEATORIAMENTE")
print("=" * 60)

print(f"ID:    {case['id']}")
print(f"TIPO:  {case['tipo']}")
print(f"INPUT: {case['input']}")

print("\nEjecutando...\n")

result = run_case(case)

print("=" * 60)
print("RESULTADO")
print("=" * 60)

print(f"Decisión:       {result['decision']}")
print(f"Validación:     {result['validation_ok']}")
print(f"Latencia:       {result['latency_seconds']} s")
print(f"Finish reason:  {result['finish_reason']}")

print("\nUSO DE TOKENS:")
print(result["usage"])

print("\nMÉTRICAS GROQ:")
print(result["metrics"])

print("\nDIAGNÓSTICO HTTP:")
print(result["http"])

print("\nSALIDA RAW:")
print(result["raw_output"])

if result["validation_errors"]:
    print("\nERRORES DE VALIDACIÓN:")
    for error in result["validation_errors"]:
        print(f"- {error}")

if result["technical_error"]:
    print("\nERROR TÉCNICO:")
    print(result["technical_error"])

CASO SELECCIONADO ALEATORIAMENTE
ID:    case_01
TIPO:  normal
INPUT: Mi computador no enciende desde esta mañana.

Ejecutando...

HTTP 429 - límite de tasa alcanzado. Reintentando en 4.00 segundos (intento 1/4)...
RESULTADO
Decisión:       OK_VALIDADO
Validación:     True
Latencia:       5.5733 s
Finish reason:  stop

USO DE TOKENS:
{'prompt_tokens': 6268, 'completion_tokens': 528, 'total_tokens': 6796}

MÉTRICAS GROQ:
{'queue_time': 0.229296277, 'prompt_time': 0.316947085, 'completion_time': 0.629682322, 'total_time': 0.946629407, 'cached_tokens': None}

DIAGNÓSTICO HTTP:
{'status_code': 200, 'url': 'https://api.groq.com/openai/v1/chat/completions', 'network_overhead_seconds': 4.6267, 'headers': {'x-groq-region': 'msp', 'cf-ray': 'a3986790bf293f15-BOG', 'x-ratelimit-limit-tokens': '8000', 'x-ratelimit-remaining-tokens': '4', 'x-ratelimit-reset-tokens': '59.97s', 'x-ratelimit-limit-requests': '1000', 'x-ratelimit-remaining-requests': '998', 'x-ratelimit-reset-requests': '2m52.8s'}, 'at

In [14]:
execution = {
    "prompt_version": PROMPT_VERSION,
    "model": MODEL,
    "integration": "direct_http_requests_session",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_cases": len(results),
    "results": results,
}

try:
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
    history = json.loads(content) if content else {}
except (json.JSONDecodeError, OSError):
    history = {}

if not isinstance(history, dict):
    history = {}

if not isinstance(history.get("executions"), list):
    history["executions"] = []

history["executions"].append(execution)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"Resultados guardados en: {RESULTS_PATH}")
print(f"Integración registrada: {execution['integration']}")
print(f"Ejecuciones almacenadas: {len(history['executions'])}")

Resultados guardados en: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/docs/results.json
Integración registrada: direct_http_requests_session
Ejecuciones almacenadas: 5
